# Explore trained encoder features

Load one fixed model, extract one deterministic center-patch vector per scan,
then explore a **PCA** projection with native Plotly. Optional UMAP is available
only if a compatible `umap-learn` is already installed; no dependency is installed
automatically. No training occurs and classifier-free models are supported.

The default projection averages longitudinal scan vectors per animal. Scan-level
points remain available for timepoint exploration, but repeated scans are not
independent animals. Metadata colors never enter the reducer. Apparent separation
is not classification performance, proof of anatomical reasoning, or a leakage
check. Cohort, condition and acquisition can be confounded.

In [ ]:
from pathlib import Path
from typing import Literal
import sys
import json

REPO_ROOT: Path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/models/networks.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display
from src.evaluation.attention import load_attention_model
from src.evaluation.feature_space import encoder_feature_table, extract_scan_features, project_features, feature_space_figure
from src.evaluation.k_fold import SubjectIndex
from src.utils.loader import RodentDatasets
from src.experiments.training import configure_runtime
from src.preprocessing.preprocessor import CorrectX10, Normalize, PadderCutter, Preprocessor, Reorient, Resample

REFERENCE: Path = REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49" / "Anat" / "TBI_fm_19_49_N4.nii.gz"
ref_img = nib.load(REFERENCE)
print("Reference axes:", nib.aff2axcodes(ref_img.affine))
runtime = configure_runtime(seed=42, memory_growth=True, mixed_precision=False)

## 1. Choose one model and layer

Use an explicit run/fold, weights plus their experiment file, or a complete
legacy export (`RUN_DIR=None`, `CHECKPOINT_KIND="model"`, explicit `MODEL_FILE`).
The default encoder feature directly precedes classifier pooling or feeds the
first decoder up-convolution. Do not mix feature vectors from different folds
or independently trained models into the same coordinate system.

In [ ]:
RUN_DIR: Path | None = REPO_ROOT / "results" / "mouse_full_3drnet" / "RUN_20261014_182511_289035"
FOLD: int = 0
MODEL_FILE: Path | None = None
EXPERIMENT_FILE: Path | None = None
CHECKPOINT_KIND: Literal["weights", "model"] = "weights"
FEATURE_LAYER: str | None = None

model, saved_config, checkpoint_file = load_attention_model(
    run_dir=RUN_DIR, fold=FOLD, model_file=MODEL_FILE, experiment_file=EXPERIMENT_FILE, checkpoint_kind=CHECKPOINT_KIND,
)
print("Checkpoint:", checkpoint_file)
display(encoder_feature_table(model=model))

## 2. Select animals and review metadata

Choose normalized cohorts or a CSV manifest with `Subject`, `Scan`, `Dataset`,
`ImagePath` and optional condition/modality/timepoint metadata. Subject IDs must
identify animals uniquely across datasets. All timepoints of each selected animal
are retained. The default samples at most 12 animals per cohort reproducibly;
set the cap to `None` for all selected animals. D4 is not included by default.

When a run is selected, saved split membership labels the checkpoint's own
training/validation/holdout animals. `not_in_saved_split` does not certify an
independent external cohort or audit pretraining exposure. Use `ROLE_FILTER` to
restrict the view. Training examples may otherwise appear; this is exploratory
visualization, not held-out evaluation. No source masks are required for CSV MRI inputs.

In [ ]:
DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
DATASETS: tuple[str, ...] = ("D1", "D3")
CSV_MANIFEST: Path | None = None
MAX_ANIMALS_PER_DATASET: int | None = 12
SEED: int = 42
ROLE_FILTER: tuple[str, ...] | None = None

if CSV_MANIFEST is None:
    loader = RodentDatasets()
    for dataset in DATASETS:
        loader.add_normalized_dataset(dataset_path=DATA_ROOT, dataset=dataset, image_name="N4")
    subjects = SubjectIndex(loader)
    manifest = subjects.manifest(subjects.subject_ids)
    modalities = {}
    for key in loader.get_subjects_list():
        scan = loader.get_subject(key)
        modalities[(key[0], key[1])] = scan.metadata.get("modality", scan.modality)
    manifest["Modality"] = [modalities[(row.Group, row.Scan)] for row in manifest.itertuples()]
else:
    manifest = pd.read_csv(CSV_MANIFEST)
    for column in ("Subject", "Scan", "Dataset", "ImagePath"):
        if column not in manifest:
            raise ValueError(f"CSV manifest requires {column}")
    image_paths = []
    for value in manifest["ImagePath"]:
        image_path = Path(value)
        if not image_path.is_absolute():
            image_path = CSV_MANIFEST.parent / image_path
        image_paths.append(str(image_path.resolve()))
    manifest["ImagePath"] = image_paths

for column in ("Condition", "Modality", "Timepoint", "Strain", "Subgroup"):
    if column not in manifest:
        manifest[column] = "unknown"
    manifest[column] = manifest[column].fillna("unknown").replace("", "unknown")
if RUN_DIR is not None:
    split_file = RUN_DIR / "splits.json"
    if split_file.is_file():
        split = json.loads(split_file.read_text(encoding="utf-8"))
        is_cv = isinstance(split, list)
        if is_cv:
            split = split[FOLD]
        roles = {}
        for role in ("train", "validation", "test", "external"):
            label = role
            if is_cv and role == "test":
                label = "holdout"
            for subject in split.get(role, []):
                if subject in roles:
                    raise ValueError("Saved animal membership overlaps roles")
                roles[subject] = label
        manifest["DatasetRole"] = manifest["Subject"].map(roles).fillna("not_in_saved_split")
if "DatasetRole" not in manifest:
    manifest["DatasetRole"] = "unknown"
if ROLE_FILTER is not None:
    manifest = manifest.loc[manifest["DatasetRole"].isin(ROLE_FILTER)].copy()
if MAX_ANIMALS_PER_DATASET is not None and MAX_ANIMALS_PER_DATASET < 1:
    raise ValueError("Use a positive animal cap or None")
rng = np.random.default_rng(SEED)
selected = []
for dataset, group in manifest.groupby("Dataset", sort=True):
    animals = sorted(group["Subject"].unique())
    if MAX_ANIMALS_PER_DATASET is not None and len(animals) > MAX_ANIMALS_PER_DATASET:
        animals = rng.choice(animals, size=MAX_ANIMALS_PER_DATASET, replace=False).tolist()
    selected.append(group.loc[group["Subject"].isin(animals)])
if not selected:
    raise ValueError("No scans match the selection")
manifest = pd.concat(selected, ignore_index=True).sort_values(["Subject", "Scan"]).reset_index(drop=True)
display(manifest)
display(manifest.groupby(["Dataset", "Condition", "Modality", "DatasetRole"]).agg(Animals=("Subject", "nunique"), Scans=("Scan", "size")).reset_index())

## 3. Make the deterministic input pipeline

Saved runs supply spacing, normalization, units and patch size. Flexible legacy
exports use the visible fallbacks. Every scan uses the common RIA reference,
linear MRI interpolation and a deterministic center crop/pad. These are global
spatial means of encoder channels from one center patch, not whole-brain features.
No augmentation, annotation targets or classifier decisions are used.

In [ ]:
LEGACY_PATCH_SIZE: tuple[int, int, int] = (80, 80, 80)
LEGACY_SPACING_MM: tuple[float, float, float] = (0.1, 0.1, 0.1)
LEGACY_NORMALIZATION: Literal["zscore", "minmax"] = "zscore"
LEGACY_CORRECT_UNITS: bool = True

patch_size = LEGACY_PATCH_SIZE
spacing_mm = LEGACY_SPACING_MM
normalization = LEGACY_NORMALIZATION
correct_units = LEGACY_CORRECT_UNITS
if saved_config is not None:
    patch_size = saved_config.preprocessing.patch_size
    spacing_mm = saved_config.preprocessing.spacing_mm
    normalization = saved_config.preprocessing.normalization
    correct_units = saved_config.preprocessing.correct_units
if all(size is not None for size in model.input_shape[1:4]):
    patch_size = tuple(int(size) for size in model.input_shape[1:4])
steps = [Reorient(ref_img=ref_img)]
if correct_units:
    steps.append(CorrectX10(reference_resolution=spacing_mm))
resampling = Resample(target_resolution=spacing_mm, interpolation=1, orthogonal=False)
normalizing = Normalize(mode=normalization)
cropping = PadderCutter(img_size=patch_size)
steps.extend([resampling, normalizing, cropping])
pipeline = Preprocessor(steps=steps)

In [ ]:
RUN_FEATURE_EXTRACTION: bool = False  # One forward pass per selected scan.

scan_features = None
if RUN_FEATURE_EXTRACTION:
    scan_features = extract_scan_features(model=model, manifest=manifest, pipeline=pipeline, layer=FEATURE_LAYER)
    print("Feature matrix:", scan_features.values.shape, "Layer:", scan_features.layer)
else:
    print("Review the selected animals, then set RUN_FEATURE_EXTRACTION=True.")

## 4. Project and explore

PCA is available with the existing dependencies. UMAP is optional and exploratory;
its apparent clusters and distances can change with settings. Both receive only
feature vectors. Metadata is attached after fitting for color/hover, never used
as supervision. Standardization can change the result and is explicit.

Animal aggregation averages scan vectors first; varying timepoints/modalities
are listed together. Use scan aggregation for longitudinal trajectories, with
the same animal identifiable in hover. Plot counts distinguish animals and scans.
These plots do not establish classifier generalization or rule out confounding.

In [ ]:
PROJECTION_METHOD: Literal["pca", "umap"] = "pca"
AGGREGATION: Literal["animal", "scan"] = "animal"
STANDARDIZE: bool = True
PROJECTION_SEED: int = 42
UMAP_NEIGHBORS: int = 10
UMAP_MIN_DIST: float = 0.1

projection = None
if scan_features is not None:
    projection = project_features(
        features=scan_features, aggregation=AGGREGATION, method=PROJECTION_METHOD,
        standardize=STANDARDIZE, seed=PROJECTION_SEED, n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST,
    )
    print(projection.parameters)
    print("PCA explained variance:", projection.explained_variance)
    display(projection.points)

In [ ]:
COLOR_BY: str = "Condition"  # Try Modality, DatasetRole, Dataset, Timepoint or Subgroup.
SYMBOL_BY: str = "Dataset"

if projection is not None:
    feature_figure = feature_space_figure(projection=projection, color=COLOR_BY, symbol=SYMBOL_BY)
    feature_figure.show(renderer="vscode")

In [ ]:
import hashlib

EXPORT_FEATURE_SPACE: bool = False
FEATURE_OUTPUT: Path | None = None  # Explicit fresh folder; no files overwritten.

if EXPORT_FEATURE_SPACE:
    if scan_features is None or projection is None or FEATURE_OUTPUT is None:
        raise ValueError("Build the projection and set FEATURE_OUTPUT to a fresh folder")
    FEATURE_OUTPUT.mkdir(parents=True, exist_ok=False)
    np.save(FEATURE_OUTPUT / "scan_features.npy", scan_features.values)
    scan_features.manifest.to_csv(FEATURE_OUTPUT / "feature_manifest.csv", index=False)
    projection.points.to_csv(FEATURE_OUTPUT / "projection.csv", index=False)
    provenance = {
        "checkpoint": str(checkpoint_file), "checkpoint_sha256": hashlib.sha256(checkpoint_file.read_bytes()).hexdigest(),
        "reference": str(REFERENCE), "reference_sha256": hashlib.sha256(REFERENCE.read_bytes()).hexdigest(),
        "projection": projection.parameters, "explained_variance": projection.explained_variance,
        "patch_size": patch_size, "spacing_mm": spacing_mm, "normalization": normalization,
        "correct_units": correct_units, "crop": "deterministic_center",
    }
    (FEATURE_OUTPUT / "feature_analysis.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    print("Feature-space files saved to:", FEATURE_OUTPUT)